# 🤖 Python 기초로 챗봇 만들기

**주제**: 이전 챕터들에서 배운 모든 것 (변수 · 함수 · 조건문 · 반복문 · 리스트 · 딕셔너리 · 파일 I/O · pandas) 을 한 프로젝트로 응용한다.

이번 챕터는 **하나의 챗봇을 점진적으로 만든다**. 한 줄짜리 API 호출에서 시작해 → 명령어 시스템 · 페르소나 · 에러 복구 · 대화 기록 분석까지 갖춘 완성형 챗봇으로 키운다.

| 섹션 | 추가되는 것 | 활용 챕터 |
|------|------------|----------|
| 1. 왜 챗봇 · 무엇이 LLM | 개념 도입 | — |
| 2. 환경 세팅 | HF 토큰 + `.env` | Ch3 |
| 3. 첫 호출 | `InferenceClient` 한 줄 | Ch1 |
| 4. 함수로 묶기 | `def ask(...)` | **Ch8** |
| 5. 대화 루프 | `while` + `input` | **Ch5** |
| 6. 입력 검증 | `if/elif` | **Ch4** |
| 7. 대화 기록 | `list` 누적 | **Ch6** |
| 8. messages 포맷 | `list[dict]` | **Ch7** |
| 9. 슬래시 명령어 | `/save`, `/load`, `/clear` | **Ch9** + Ch4 |
| 10. 페르소나 | `dict` 로 봇 성격 전환 | Ch7 + Ch4 |
| 11. 에러 처리 | `try/except` + 재시도 | Ch4 + Ch5 |
| 12. 대화 데이터 분석 | pandas + Counter | **Ch10** + Ch7 |

> 💡 **이 노트북은 점진적이다.** 위에서 아래로 실행하면 `chat()` 함수가 한 단계씩 발전한다.
> 마지막 셀까지 가면 완성형 챗봇이 있다. 같은 결과의 단일 파일 버전은 `chatbot.py` 참고.


---
## 1. 왜 챗봇 · 무엇이 LLM

### 1.1 챗봇 = 입력 받고 → 응답 출력 → 반복

가장 단순화하면 챗봇은 이렇다.

```python
while True:
    질문 = input()
    응답 = 어딘가에서_답을_가져온다(질문)
    print(응답)
```

이전 챕터에서 본 `input()` + `while` 그대로다. 핵심은 **"어딘가에서 답을 가져오는"** 부분 — 우리는 거기에 LLM API 를 끼워넣을 것이다.

### 1.2 LLM (Large Language Model) 한 줄 정의

- 인터넷 규모의 텍스트로 학습된 거대 신경망.
- 입력 → 다음에 올 가장 그럴듯한 단어를 확률적으로 생성.
- 예: ChatGPT, Claude, Gemini, Llama, Mistral …

### 1.3 우리가 쓸 것 — HuggingFace Inference API

HuggingFace 는 수만 개 모델을 공개 호스팅하는 플랫폼이다. **무료 계정** 으로도 여러 모델을 API 로 호출할 수 있다.

| 구분 | OpenAI / Anthropic | HuggingFace |
|------|-------------------|-------------|
| 비용 | 거의 무조건 유료 | **무료 티어 있음** |
| 모델 선택 | 자사 모델만 | 수만 개 (Llama, Mistral, Qwen …) |
| 코드 형태 | 거의 동일 (OpenAI 호환 패턴) | 거의 동일 |

이 챕터에서는 HuggingFace 만 다루지만, **다른 API 도 코드 패턴은 거의 같다**. 한 번 익히면 어디든 적용된다.


---
## 2. 환경 세팅 — 토큰 발급 + 라이브러리 설치

📌 **활용**: Ch3 (모듈), Ch9 (`.env` 파일)

### 2.1 HuggingFace 토큰 발급

1. https://huggingface.co/join → 가입 (이메일만)
2. https://huggingface.co/settings/tokens → **+ New token**
3. Type: **Read** (이 챕터에는 충분)
4. 토큰 문자열 (`hf_...`) **복사 — 한 번만 표시됨**

### 2.2 라이브러리 설치

```bash
pip install huggingface-hub python-dotenv
```

### 2.3 토큰을 `.env` 파일에 보관

소스 코드에 토큰을 직접 쓰면 **GitHub 푸시 시 유출** 된다. `.env` 파일에 분리한다.

`.env`:
```
HF_TOKEN=hf_여기에본인토큰
```

`.gitignore` 에 반드시 `.env` 추가:
```
.env
```

> 🚨 **실수로 토큰을 push 했다면** 즉시 HuggingFace 에서 **revoke** 후 새로 발급한다. Ch11 §10 참고.


In [ ]:
# .env 파일에서 환경변수 로드
import os
from dotenv import load_dotenv

load_dotenv()              # .env 파일을 읽어 os.environ 에 주입

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    print("❌ HF_TOKEN 이 설정되지 않았습니다. .env 파일 확인.")
else:
    print(f"✅ 토큰 로드됨: {HF_TOKEN[:6]}...{HF_TOKEN[-4:]}")

### 2.4 사용할 모델 정하기

이번 챕터에서는 **무료 inference 풀** 에서 잘 작동하는 모델을 쓴다. 모델 선택은 변수 하나로 끝나니, 안 되면 다른 걸로 바꿔본다.

| 모델 ID | 크기 | 특징 |
|---------|-----|------|
| `meta-llama/Llama-3.2-3B-Instruct` | 3B | 한국어 OK, 빠름 |
| `Qwen/Qwen2.5-7B-Instruct`         | 7B | 한국어 우수 |
| `microsoft/Phi-3.5-mini-instruct`  | 3.8B | 영어 중심 |
| `HuggingFaceH4/zephyr-7b-beta`     | 7B | 안정적 |

> ⚠️ 일부 모델은 **liaison page 동의** 가 필요하다. 모델 페이지에서 한 번 클릭해주면 끝.


In [ ]:
# 사용할 모델 — 여기 하나만 바꾸면 전체 챗봇에 반영됨
MODEL = "meta-llama/Llama-3.2-3B-Instruct"

---
## 3. 첫 API 호출 — 한 줄로 답변 받기

📌 **활용**: Ch1 (변수/문자열), Ch3 (모듈)

`InferenceClient` 가 HuggingFace API 와의 대화를 도와준다.


In [ ]:
from huggingface_hub import InferenceClient

client = InferenceClient(token=HF_TOKEN)

# 가장 단순한 형태 — 한 번 묻고 한 번 받기
response = client.chat_completion(
    messages=[{"role": "user", "content": "안녕! 너는 누구니? 한 문장으로 소개해줘."}],
    model=MODEL,
    max_tokens=100,
)

# 응답 객체에서 텍스트만 꺼내기
answer = response.choices[0].message.content
print(answer)

### 3.1 응답 객체의 구조

응답은 dict-like 객체로 온다 (Ch7 의 딕셔너리 응용).

```
response
├── choices: [응답 후보 리스트]
│   └── [0]
│       └── message
│           ├── role:    "assistant"
│           └── content: "안녕하세요! 저는 ..."
└── usage: 토큰 사용량
```

`.choices[0].message.content` 가 우리가 원하는 답이다.


In [ ]:
# 응답 전체 살펴보기
print("─" * 40)
print(f"role    : {response.choices[0].message.role}")
print(f"content : {response.choices[0].message.content}")
print("─" * 40)
print(f"사용 토큰 : 입력 {response.usage.prompt_tokens} + 출력 {response.usage.completion_tokens}")

---
## 4. 재사용 가능한 함수로 묶기

📌 **활용**: **Ch8 (함수)**

같은 코드를 반복 쓸 거니까 함수로 만든다. 매개변수와 디폴트 인자(Ch8 §6) 도 활용.


In [ ]:
def ask(question: str, max_tokens: int = 200, model: str = MODEL) -> str:
    """질문 한 개를 보내고 답을 문자열로 반환.

    Args:
        question  : 사용자 질문 (str)
        max_tokens: 최대 응답 토큰 수 (기본 200)
        model     : 사용할 모델 ID (기본 전역 MODEL)

    Returns:
        모델 응답 텍스트 (str)
    """
    response = client.chat_completion(
        messages=[{"role": "user", "content": question}],
        model=model,
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content


# 이제 한 줄로 질문 가능
print(ask("파이썬 리스트와 튜플의 가장 큰 차이는?"))
print()
print(ask("위 답을 한 문장으로 요약해줘.", max_tokens=50))

> 💡 **함수의 가치** — 같은 일을 여러 번 시켜야 할 때, 함수가 없으면 위 셀의 4줄을 매번 복사해야 한다. 함수로 묶으면 `ask("...")` 한 줄.

이게 Ch8 에서 본 **DRY(Don't Repeat Yourself)** 원칙의 실전 예시다.


---
## 5. 대화 루프 — 챗봇의 본체

📌 **활용**: **Ch5 (반복문)**, Ch3 (`input`), Ch4 (`if`)

지금까지는 한 번 묻고 끝났다. 이제 **계속 묻고 답하기** 위해 `while` 루프를 쓴다. 종료 명령으로 빠져나오는 `break` 가 핵심.


In [ ]:
def chat_v1():
    """첫 챗봇 — 무한 루프로 묻고 답한다. '/quit' 입력 시 종료."""
    print("🤖 챗봇 시작! 종료하려면 /quit")
    while True:
        question = input("\n나 > ")
        if question == "/quit":
            print("👋 안녕!")
            break
        answer = ask(question)
        print(f"봇 > {answer}")


# 셀에서 실행 — input() 으로 입력 받음
# chat_v1()    # ← 주석 풀고 실행

> ⚠️ `input()` 은 Jupyter 셀에서 잘 동작하지만 **셀 위쪽 입력창** 으로 뜨니 놓치지 말 것.
> 실행이 멈춰있는 것처럼 보이면 입력창 확인.

### 5.1 약점 — 이 챗봇의 문제 3가지

1. **빈 입력**을 그대로 모델에 보냄 → 비효율
2. **이전 대화를 기억 못 함** → "내가 방금 뭘 물었지?" 못 함
3. **종료 명령** 이 `/quit` 하나뿐

다음 섹션부터 하나씩 해결한다.


---
## 6. 입력 검증 — 빈 입력 / 너무 긴 입력

📌 **활용**: **Ch4 (조건문)**, Ch2 (비교 연산), Ch1 (`strip`, `len`)

`if/elif` 로 잘못된 입력을 거른다.


In [ ]:
def chat_v2():
    print("🤖 챗봇 v2. /quit 으로 종료")
    while True:
        question = input("\n나 > ").strip()        # 양 끝 공백 제거 (Ch1)

        # 종료 명령
        if question == "/quit":
            print("👋 안녕!")
            break

        # 빈 입력 → 무시하고 다음 루프
        if not question:                            # Ch2 — 빈 문자열은 falsy
            print("(빈 입력은 보내지 않습니다)")
            continue                                # Ch5 — 다음 반복으로

        # 너무 긴 입력 → 거부
        if len(question) > 1000:                    # Ch1 — len
            print(f"⚠️ 너무 깁니다 ({len(question)}자). 1000자 이하로.")
            continue

        # 통과 → 모델에 전달
        answer = ask(question)
        print(f"봇 > {answer}")


# chat_v2()

**`if` / `continue` / `break` 의 조합** 이 깔끔한 챗봇 루프의 표준 패턴이다.

- `continue` : 이번 입력은 건너뛰고 **다음 입력** 받기
- `break`    : 루프 자체를 **종료**


---
## 7. 대화 기록 — 모델이 기억하게 하기 (1차 시도)

📌 **활용**: **Ch6 (리스트)**

지금까지의 챗봇은 매 질문이 독립적이라 "내 이름이 뭐였지?" 같은 후속 질문에 답을 못 한다. 해결: **이전 대화 전부를 매번 같이 보낸다**.

먼저 가장 단순한 방법 — 그냥 문자열을 list 에 쌓아 합쳐서 보내기.


In [ ]:
def chat_v3_naive():
    """순진한 기록 방식 — 모든 대화를 하나의 긴 문자열로."""
    history = []                                     # Ch6 — 리스트

    while True:
        question = input("\n나 > ").strip()
        if question == "/quit": break
        if not question: continue

        # 이전 모든 대화 + 새 질문을 하나의 prompt 로
        history.append(f"User: {question}")
        prompt = "\n".join(history)                # Ch6 — 리스트 → 문자열

        answer = ask(prompt, max_tokens=300)
        print(f"봇 > {answer}")
        history.append(f"Assistant: {answer}")


# chat_v3_naive()

### 7.1 이 방식의 문제

- "User:", "Assistant:" 같은 prefix 를 모델이 **메타로 인식 못 함**
- 토큰을 너무 많이 씀 (전체 텍스트 매번 재전송)
- 페르소나(system prompt) 적용이 어려움

→ 다음 섹션에서 **정식 포맷** 으로 해결한다.


---
## 8. messages 포맷 = `list[dict]`

📌 **활용**: **Ch7 (딕셔너리)**, Ch6 (리스트)

OpenAI / HuggingFace / Anthropic 모두 **공통 포맷** 을 쓴다.

```python
messages = [
    {"role": "system",    "content": "당신은 친절한 AI 입니다."},
    {"role": "user",      "content": "안녕!"},
    {"role": "assistant", "content": "안녕하세요!"},
    {"role": "user",      "content": "오늘 뭐 할까?"},
]
```

- `system` : 봇의 성격 / 규칙. **맨 앞 한 번**.
- `user` : 사용자 발화.
- `assistant` : 봇의 이전 응답.

이게 Ch6(리스트) + Ch7(딕셔너리) 의 응용. 리스트 안에 딕셔너리들.


In [ ]:
def chat_v4():
    """제대로 된 대화 기록 — messages 포맷 사용."""
    messages = [
        {"role": "system",
         "content": "당신은 친절하고 간결한 한국어 AI 비서입니다."},
    ]

    while True:
        question = input("\n나 > ").strip()
        if question == "/quit": break
        if not question: continue

        # 사용자 메시지를 history 에 추가
        messages.append({"role": "user", "content": question})

        # 전체 messages 를 보내고 응답 받기
        response = client.chat_completion(
            messages=messages,
            model=MODEL,
            max_tokens=400,
        )
        reply = response.choices[0].message.content

        # 봇 응답도 history 에 추가 — 다음 턴에 컨텍스트로 사용됨
        messages.append({"role": "assistant", "content": reply})
        print(f"봇 > {reply}")


# chat_v4()
# 이제 "내 이름은 길동이야" → "내 이름이 뭐였지?" 가 통한다!

### 8.1 dict 의 `append` 가 어떻게 작동하나

```python
messages = [{"role": "system", "content": "..."}]
messages.append({"role": "user", "content": "안녕"})   # Ch6 list.append
messages[-1]["content"]                                  # Ch7 dict 접근
```

리스트에는 임의의 객체를 넣을 수 있다 (Ch6). 우리는 딕셔너리(Ch7)를 넣고, `[-1]["content"]` 처럼 양쪽 문법을 조합해 쓴다.

### 8.2 토큰 절약 팁 — history 자르기

오래된 대화를 무한히 쌓으면 토큰 비용이 폭증한다. 보통 **최근 N 턴** 만 유지:

```python
MAX_TURNS = 10   # user 10번 + assistant 10번 + system 1 = 21개 까지
if len(messages) > 1 + 2 * MAX_TURNS:
    # system 은 보존, 가장 오래된 user-assistant 쌍 제거
    messages = [messages[0]] + messages[-2 * MAX_TURNS:]
```


---
## 9. 슬래시 명령어 — `/save`, `/load`, `/clear`

📌 **활용**: **Ch9 (파일 I/O, JSON)**, Ch4 (`if/elif`)

대화를 파일로 저장하고 다시 불러오는 명령어를 추가한다.


In [ ]:
import json
from pathlib import Path


def chat_v5():
    """슬래시 명령어 시스템 추가."""
    messages = [{"role": "system",
                 "content": "당신은 친절한 한국어 AI 비서입니다."}]

    HELP = """
사용 가능한 명령어:
  /quit              종료
  /clear             대화 기록 초기화 (system 만 남김)
  /save [파일명]      대화를 JSON 으로 저장 (기본: chat.json)
  /load 파일명        저장된 대화 불러오기
  /history           현재 메시지 개수 표시
  /help              이 도움말
"""

    print("🤖 챗봇 v5. /help 로 명령어 확인.")
    while True:
        q = input("\n나 > ").strip()
        if not q: continue

        # ─── 슬래시 명령 처리 ──────────────────────────────────
        if q == "/quit":
            print("👋 안녕!"); break

        elif q == "/help":
            print(HELP)
            continue

        elif q == "/clear":
            messages = messages[:1]               # system 만 남김 (Ch6 슬라이싱)
            print("🗑️  대화 기록 초기화")
            continue

        elif q == "/history":
            print(f"📜 {len(messages)} 개 메시지 (system 1 + 대화 {len(messages)-1})")
            continue

        elif q.startswith("/save"):
            # "/save filename" 또는 "/save"
            parts = q.split(maxsplit=1)           # Ch1 — split
            filename = parts[1] if len(parts) > 1 else "chat.json"
            with open(filename, "w", encoding="utf-8") as f:   # Ch9
                json.dump(messages, f, ensure_ascii=False, indent=2)
            print(f"💾 저장됨: {filename}")
            continue

        elif q.startswith("/load"):
            parts = q.split(maxsplit=1)
            if len(parts) < 2:
                print("⚠️  사용법: /load 파일명")
                continue
            filename = parts[1]
            if not Path(filename).exists():       # Ch9 + Ch4
                print(f"⚠️  파일 없음: {filename}")
                continue
            with open(filename, encoding="utf-8") as f:
                messages = json.load(f)
            print(f"📂 불러옴: {filename} ({len(messages)} 메시지)")
            continue

        elif q.startswith("/"):
            print(f"⚠️  모르는 명령: {q}.  /help 참고")
            continue

        # ─── 일반 대화 ──────────────────────────────────────
        messages.append({"role": "user", "content": q})
        response = client.chat_completion(
            messages=messages, model=MODEL, max_tokens=400,
        )
        reply = response.choices[0].message.content
        messages.append({"role": "assistant", "content": reply})
        print(f"봇 > {reply}")


# chat_v5()

### 9.1 디스패치 패턴

위의 `if/elif/elif/...` 흐름이 명령어 디스패치(dispatch) 의 가장 직관적인 형태다. 명령어가 많아지면 **딕셔너리 디스패치** 로 깔끔하게 정리할 수도 있다:

```python
def cmd_clear(state): ...
def cmd_save(state, args): ...

COMMANDS = {
    "/clear": cmd_clear,
    "/save":  cmd_save,
}

handler = COMMANDS.get(name)
if handler: handler(...)
```

→ 이건 연습문제로 남긴다.


---
## 10. 페르소나 — 봇 성격 전환

📌 **활용**: Ch7 (dict), Ch4 (if)

`system` prompt 를 통째로 교체하면 봇의 성격이 바뀐다. 여러 페르소나를 딕셔너리로 관리.


In [ ]:
PERSONAS = {
    "default":  "당신은 친절하고 간결한 한국어 AI 비서입니다.",
    "tutor":    "당신은 파이썬 초보자를 가르치는 친절한 선생님입니다. "
                "모든 답변에 짧은 코드 예시를 포함하세요.",
    "reviewer": "당신은 깐깐한 코드 리뷰어입니다. "
                "사용자가 보여주는 코드의 버그, 스타일, 개선점을 짚어주세요.",
    "comedian": "당신은 코미디언입니다. 모든 답변을 농담과 함께 하세요.",
    "tsundere": "당신은 츤데레 캐릭터입니다. 도와주면서도 시큰둥하게 답하세요. "
                "그래도 정보는 정확하게.",
}


def chat_v6():
    persona = "default"
    messages = [{"role": "system", "content": PERSONAS[persona]}]

    HELP = """명령어: /quit /clear /save /load /history /help
페르소나: /persona 이름   (사용 가능: """ + ", ".join(PERSONAS) + ")"

    print(f"🤖 챗봇 v6 [{persona}]. /help")
    while True:
        q = input(f"\n나 [{persona}] > ").strip()
        if not q: continue

        if q == "/quit": print("👋 안녕!"); break
        if q == "/help": print(HELP); continue
        if q == "/clear":
            messages = [{"role": "system", "content": PERSONAS[persona]}]
            print("🗑️  초기화"); continue

        if q.startswith("/persona"):
            parts = q.split(maxsplit=1)
            if len(parts) < 2:
                print(f"현재: {persona}.  사용 가능: {list(PERSONAS)}")
                continue
            name = parts[1]
            if name not in PERSONAS:
                print(f"⚠️  없는 페르소나. 사용 가능: {list(PERSONAS)}")
                continue
            persona = name
            messages = [{"role": "system", "content": PERSONAS[persona]}]
            print(f"🎭 전환: {persona}")
            continue

        if q.startswith("/"):
            print(f"⚠️  모르는 명령"); continue

        messages.append({"role": "user", "content": q})
        response = client.chat_completion(messages=messages, model=MODEL, max_tokens=400)
        reply = response.choices[0].message.content
        messages.append({"role": "assistant", "content": reply})
        print(f"봇 > {reply}")


# chat_v6()
# /persona tutor 하면 갑자기 친절한 선생님으로 바뀜

> 💡 페르소나는 **딕셔너리의 좋은 활용 예** 다. 새 페르소나를 추가하려면 `PERSONAS["새이름"] = "..."` 한 줄.
> 만약 if/elif 로 다 짰다면 페르소나 하나 추가할 때마다 코드를 수정해야 한다 (Ch7 의 dict-vs-if 비교).


---
## 11. 에러 처리 — 네트워크 실패, 재시도

📌 **활용**: Ch4 (`try/except`), Ch5 (`for` + `range`), Ch3 (`time.sleep`)

API 호출은 실패할 수 있다 (네트워크 끊김, rate limit, 서버 다운). 그대로 두면 챗봇이 죽는다. **try/except + 재시도** 가 표준.


In [ ]:
import time


def ask_safe(messages, model=MODEL, retries=3, base_delay=1.0):
    """호출 실패 시 지수 백오프로 재시도.

    base_delay 가 1, 재시도가 3이면 대기는 1, 2, 4초.
    """
    last_error = None
    for attempt in range(retries):                    # Ch5 — for + range
        try:
            response = client.chat_completion(
                messages=messages, model=model, max_tokens=400,
            )
            return response.choices[0].message.content
        except Exception as e:                        # Ch4 — try/except
            last_error = e
            wait = base_delay * (2 ** attempt)        # 1, 2, 4초 …
            print(f"⚠️  [{attempt+1}/{retries}] 실패: {type(e).__name__}. "
                  f"{wait:.1f}초 후 재시도…")
            time.sleep(wait)
    # 모두 실패
    return f"❌ {retries}회 재시도 후 실패: {last_error}"


# 시험: 잘못된 모델 ID 로 호출해 강제 실패 → 재시도 동작 확인
# print(ask_safe([{"role":"user","content":"hi"}], model="bogus/nonexistent"))

### 11.1 지수 백오프 (exponential backoff) 가 뭐고 왜?

| 시도 | 대기 | 누적 |
|-----|------|-----|
| 1   | 1초  | 1초 |
| 2   | 2초  | 3초 |
| 3   | 4초  | 7초 |
| 4   | 8초  | 15초 |

서버가 잠시 과부하면 짧게 기다리고, 계속 실패하면 점점 길게 기다린다. 모든 클라이언트가 동시에 즉시 재시도하면 서버를 더 죽이니까.

### 11.2 자주 마주치는 예외

| 예외 | 원인 | 대처 |
|------|-----|------|
| `HTTPError 429` | rate limit 초과 | 잠시 대기 |
| `HTTPError 401` | 토큰 잘못됨 | `.env` 확인 |
| `HTTPError 503` | 모델 로딩 중 | 잠시 후 재시도 (`x-wait-for-model`) |
| `ConnectionError` | 인터넷 단절 | 네트워크 확인 |


---
## 12. 대화 데이터 분석 — pandas + Counter

📌 **활용**: **Ch10 (pandas)**, Ch7 (Counter ≈ dict), Ch9 (file I/O)

`/save` 로 쌓인 JSON 들을 모아서 "내가 가장 많이 한 질문 단어는?" "평균 메시지 길이는?" 등을 분석한다.


In [ ]:
# 시연용 — 가상의 저장된 대화 3개를 만든다 (실제로는 /save 로 생성됨)
import json
from pathlib import Path

demo_chats = [
    [
        {"role": "system",    "content": "당신은 친절한 비서입니다."},
        {"role": "user",      "content": "파이썬에서 리스트와 튜플의 차이는?"},
        {"role": "assistant", "content": "리스트는 변경 가능, 튜플은 불변입니다."},
        {"role": "user",      "content": "예시 코드 보여줘"},
        {"role": "assistant", "content": "lst = [1,2,3]; lst[0]=9  # OK ..."},
    ],
    [
        {"role": "system",    "content": "당신은 친절한 비서입니다."},
        {"role": "user",      "content": "딕셔너리에서 값을 안전하게 가져오는 방법?"},
        {"role": "assistant", "content": "d.get(key, default) 를 쓰세요."},
    ],
    [
        {"role": "system",    "content": "당신은 친절한 비서입니다."},
        {"role": "user",      "content": "for 와 while 의 차이를 알려줘"},
        {"role": "assistant", "content": "for 는 횟수 기반, while 은 조건 기반입니다."},
        {"role": "user",      "content": "어떤걸 쓰는게 좋아?"},
        {"role": "assistant", "content": "리스트 순회면 for, 조건 만족까지면 while."},
    ],
]

for i, chat in enumerate(demo_chats, start=1):
    with open(f"chat_demo_{i}.json", "w", encoding="utf-8") as f:
        json.dump(chat, f, ensure_ascii=False, indent=2)

print("✅ 데모 파일 3개 생성")

In [ ]:
import pandas as pd
from pathlib import Path
from collections import Counter

# 저장된 모든 chat_*.json 모으기
rows = []
for path in sorted(Path('.').glob('chat_*.json')):
    with open(path, encoding='utf-8') as f:
        messages = json.load(f)
    for m in messages:
        rows.append({
            'file':    path.name,
            'role':    m['role'],
            'content': m['content'],
            'length':  len(m['content']),    # Ch1 — len
        })

df = pd.DataFrame(rows)
print(df)

In [ ]:
# 분석 1 — 역할별 메시지 수 / 평균 길이
summary = df.groupby('role')['length'].agg(['count', 'mean', 'max'])
print(summary)

In [ ]:
# 분석 2 — 사용자 질문에서 자주 쓴 단어 Top 10
user_msgs = df[df['role'] == 'user']
all_text = ' '.join(user_msgs['content']).lower()
words = all_text.replace(',', ' ').replace('.', ' ').replace('?', ' ').split()

# 너무 짧은 단어 / 흔한 조사 제외
STOPWORDS = {'은', '는', '이', '가', '을', '를', '의', '와', '과', '도', '에',
             '에서', '으로', '로', '이런', '저런', '그', '이', '저', '는데'}
words = [w for w in words if len(w) > 1 and w not in STOPWORDS]

top10 = Counter(words).most_common(10)
print('자주 쓴 단어 Top 10:')
for w, n in top10:
    print(f'  {n:3d}회  {w}')

In [ ]:
# 분석 3 — 대화별(파일별) 길이
per_chat = df.groupby('file').agg(
    messages=('role', 'count'),
    total_chars=('length', 'sum'),
).sort_values('total_chars', ascending=False)
print(per_chat)

### 12.1 응용 — 자기 대화 패턴 발견하기

이 분석을 자기 대화에 적용하면:
- 어떤 주제로 가장 많이 물어보나
- 질문이 점점 길어지나 짧아지나
- 어느 페르소나를 가장 많이 쓰나

→ Ch10 (pandas) + Ch7 (Counter) + Ch9 (file I/O) 의 자연스러운 응용.


---
## ✏️ 연습문제

**Q1.** `chat_v6` 에 자기만의 페르소나 3개 추가해보자. 예: "한 줄로만 답하는 봇", "꼭 반말로 답하는 봇", "이모지를 많이 쓰는 봇".

**Q2.** 토큰 절약 — `chat_v6` 에 §8.2 의 "최근 N 턴만 유지" 로직을 추가하라. `/maxturns 5` 명령으로 N 도 바꿀 수 있게.

**Q3.** **요약봇** — `/summarize` 명령을 추가하라. 누르면 지금까지의 대화를 봇에게 보내 "위 대화를 3문장으로 요약" 시키고, 그 요약으로 history 를 교체 (오래된 turn 제거).

**Q4.** **딕셔너리 디스패치** — §9.1 의 힌트대로 `if/elif/elif/...` 로 된 명령어 처리를 `COMMANDS = {"/clear": cmd_clear, ...}` 형태로 리팩토링하라.

**Q5.** **로그 분석 확장** — §12 의 분석에 다음을 추가:
1. 시간대별 (저장된 파일의 modification time) 대화 횟수
2. assistant 응답 중 코드 블록 (```...```) 이 들어있는 비율
3. 가장 긴 단일 응답

---

## 🎯 핵심 요약 — 이번 챕터가 챕터 1~10 을 어떻게 다 썼나

| 챕터 | 어디서 |
|------|--------|
| Ch1 — 변수/문자열 | f-string 출력, `strip`, `len`, slicing |
| Ch2 — 연산자 | `len(q) > 1000`, `not q` |
| Ch3 — 입력/모듈 | `input()`, `import` |
| Ch4 — 조건문 | 명령어 디스패치, `try/except` |
| Ch5 — 반복문 | 챗 루프, 재시도 루프, `continue/break` |
| Ch6 — 리스트 | `messages.append`, history slicing |
| Ch7 — 딕셔너리 | `{"role": ..., "content": ...}`, PERSONAS, Counter |
| Ch8 — 함수 | `ask`, `ask_safe`, `chat_vN` |
| Ch9 — 파일 I/O | `/save`, `/load`, `.env` |
| Ch10 — pandas | 대화 데이터 분석 |

여기까지 따라왔다면, **Python 기초로 진짜 동작하는 제품을 만드는 감각** 이 생긴다.

---

🎉 **시리즈 진짜 완결!** 다음 단계는 `chatbot.py` 를 자기 컴퓨터 터미널에서 실행해보고, 자기 페르소나를 만들어보는 것.
